In [3]:
import sys
from pathlib import Path

import joblib

# Ensure local modules are importable in VS Code notebooks.
cwd = Path.cwd()
candidate_dirs = [cwd, cwd / "freight_cost_prediction"]
for candidate in candidate_dirs:
    if (candidate / "data_preprocessing.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from data_preprocessing import load_vendor_invoice_data, prepare_features, split_data
from model_evaluation import (
    train_linear_regression,
    train_decision_tree,
    train_random_forest,
    evaluate_model
)

def main():
    # Handle both common notebook working directories: workspace root and this folder.
    db_candidates = [Path("data/inventory.db"), Path("../data/inventory.db")]
    db_path = next((str(p) for p in db_candidates if p.exists()), "data/inventory.db")

    model_dir = Path("models")
    model_dir.mkdir(exist_ok=True)

    # Load data
    df = load_vendor_invoice_data(db_path)

    # Prepare data
    X, y = prepare_features(df)
    X_train, X_test, y_train, y_test = split_data(X, y)

    # Train models
    lr_model = train_linear_regression(X_train, y_train)
    dt_model = train_decision_tree(X_train, y_train)
    rf_model = train_random_forest(X_train, y_train)

    # Evaluate models
    results = []
    results.append(evaluate_model(lr_model, X_test, y_test, "Linear Regression"))
    results.append(evaluate_model(dt_model, X_test, y_test, "Decision Tree Regression"))
    results.append(evaluate_model(rf_model, X_test, y_test, "Random Forest Regression"))

    # Select best model (lowest MAE)
    best_model_info = min(results, key=lambda x: x["mae"])
    best_model_name = best_model_info["model_name"]

    best_model = {
        "Linear Regression": lr_model,
        "Decision Tree Regression": dt_model,
        "Random Forest Regression": rf_model
    }[best_model_name]
    
    # Save best model
    model_path = model_dir / "predict_freight_model.pkl"
    joblib.dump(best_model, model_path)
    
    print(f"\nBest model saved: {best_model_name}")
    print(f"Model path: {model_path}")

if __name__ == "__main__":
    main()


Linear Regression Performance:
MAE  : 24.11
RMSE : 124.72
R^2  : 96.99%

Decision Tree Regression Performance:
MAE  : 32.97
RMSE : 150.31
R^2  : 95.63%

Random Forest Regression Performance:
MAE  : 26.13
RMSE : 134.79
R^2  : 96.48%

Best model saved: Linear Regression
Model path: models\predict_freight_model.pkl


In [12]:
import os
os.getcwd()

'C:\\Users\\pc\\OneDrive\\Desktop\\New folder\\freight_cost_prediction'